In [ ]:
FONTES_SEPREF = ["santos_sepref"]
CAMPOS_SEPREF = [
    "agendamento",
    "bairro_interessado",
    "bairro_ocorrencia",
    "canal",
    "cpf",
    "nome",
    "nome_logradouro",
    "numero_imovel",
    "tipo_logradouro",
]

In [ ]:
from pyspark.sql.functions import col, first, spark_max

df_sol = (
    spark.table("silver_fato_solicitacoes")
    .filter(col("fonte").isin(FONTES_SEPREF))
)

df_pivot = (
    spark.table("silver_fato_campos")
    .filter(col("fonte").isin(FONTES_SEPREF))
    .filter(col("campo").isin(CAMPOS_SEPREF))
    .groupBy("id_os")
    .pivot("campo", CAMPOS_SEPREF)
    .agg(first("valor"))
)

df_etapas = (
    spark.table("silver_fato_etapas")
    .filter(col("fonte").isin(FONTES_SEPREF))
    .groupBy("id_os")
    .agg(
        spark_max("etapa").alias("etapa_atual"),
        spark_max("data_fim_etapa").alias("data_fim_ultima_etapa"),
    )
)

df_gold = (
    df_sol
    .join(df_pivot, "id_os", "left")
    .join(df_etapas, "id_os", "left")
)

# df_gold.write.mode("overwrite").format("delta").saveAsTable("gold_fato_solicitacoes_sepref")